# 02. Monte Carlo Methods & MCMC (Markov Chain Monte Carlo)

**How random sampling solves impossible high-dimensional integrals in Bayesian ML, Generative AI, and Diffusion Models.**

---

## 1. What is Monte Carlo Integration?

In many machine learning problems, computing continuous expectations $\mathbb{E}_{p}[f(X)] = \int f(x) p(x) dx$ analytically is impossible because $p(x)$ is high-dimensional or non-integrable.

By the **Law of Large Numbers (LLN)**, we approximate the integral by averaging $N$ random samples drawn from $p(x)$:

$$\mathbb{E}[f(X)] = \int f(x) p(x) dx \approx \frac{1}{N} \sum_{i=1}^N f(x_i) \quad \text{where } x_i \sim p(x)$$

- **Convergence Rate**: Standard error decreases at rate $O\left(\frac{1}{\sqrt{N}}\right)$, **independent of the dimension $D$**! (Beating the curse of dimensionality!).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estimating Pi using Monte Carlo Sampling
N = 10000
x = np.random.uniform(-1, 1, size=N)
y = np.random.uniform(-1, 1, size=N)

# Inside unit circle: x^2 + y^2 <= 1
inside_circle = (x**2 + y**2) <= 1.0
pi_estimate = 4.0 * np.sum(inside_circle) / N

print(f"Monte Carlo Pi Estimate ({N} samples): {pi_estimate:.5f} (True Pi: {np.pi:.5f})")

# Visualizing
plt.figure(figsize=(6, 6))
plt.scatter(x[inside_circle], y[inside_circle], color='teal', s=1, alpha=0.5, label='Inside Circle')
plt.scatter(x[~inside_circle], y[~inside_circle], color='coral', s=1, alpha=0.5, label='Outside')
plt.title(r"Monte Carlo Estimation of $\pi \approx 4 \times \frac{N_{inside}}{N_{total}}$")
plt.axis('equal')
plt.legend()
plt.show()


---

## 2. Rejection Sampling & Importance Sampling

- **Rejection Sampling**: Sample from a simpler proposal distribution $q(x)$ such that $p(x) \leq M q(x)$, and accept with probability $\frac{p(x)}{M q(x)}$.
- **Importance Sampling**: When sampling from $p(x)$ is impossible, sample from $q(x)$ and weight each sample by the **importance weight** $w(x) = \frac{p(x)}{q(x)}$:
  $$\mathbb{E}_p[f(X)] = \int f(x) \frac{p(x)}{q(x)} q(x) dx \approx \frac{1}{N} \sum_{i=1}^N f(x_i) \frac{p(x_i)}{q(x_i)} \quad (x_i \sim q(x))$$


---

## 3. Markov Chain Monte Carlo (MCMC) & Metropolis-Hastings

### The Challenge:
In Bayesian statistics and energy-based generative models, we often know the unnormalized probability density $\tilde{p}(x)$, but the normalizing partition function $Z = \int \tilde{p}(x) dx$ is intractable:
$$p(x) = \frac{\tilde{p}(x)}{Z}$$

### The Metropolis-Hastings Algorithm:
Construct a Markov chain whose stationary distribution is exactly $p(x)$!
1. Start at state $x_t$.
2. Propose a new candidate $x' \sim q(x' \mid x_t)$.
3. Compute the **Acceptance Ratio $\alpha$** (notice $Z$ cancels out!):
   $$\alpha = \min\left(1, \frac{p(x') q(x_t \mid x')}{p(x_t) q(x' \mid x_t)} \right) = \min\left(1, \frac{\tilde{p}(x') q(x_t \mid x')}{\tilde{p}(x_t) q(x' \mid x_t)} \right)$$
4. With probability $\alpha$, accept $x_{t+1} = x'$; otherwise reject and set $x_{t+1} = x_t$.


In [ ]:
# Metropolis-Hastings Sampling from an Unnormalized Bimodal Distribution
# p_tilde(x) = exp(-0.5*(x-2)^2) + 0.7*exp(-0.5*(x+2)^2)
def unnormalized_target(x):
    return np.exp(-0.5 * (x - 2.5)**2) + 0.7 * np.exp(-0.5 * (x + 2.5)**2)

# MCMC Sampling loop
num_mcmc_samples = 20000
samples = [0.0]
current_x = 0.0

for _ in range(num_mcmc_samples):
    # Gaussian random walk proposal: q(x' | x) = N(x, 1)
    proposed_x = current_x + np.random.normal(0, 1.0)
    
    # Acceptance ratio
    alpha = min(1.0, unnormalized_target(proposed_x) / unnormalized_target(current_x))
    
    # Accept or reject
    if np.random.rand() < alpha:
        current_x = proposed_x
    samples.append(current_x)

samples = np.array(samples[2000:]) # Discard burn-in

# Plot histogram of MCMC samples vs true density curve
x_grid = np.linspace(-6, 6, 300)
plt.figure(figsize=(9, 5))
plt.hist(samples, bins=60, density=True, alpha=0.6, color='skyblue', label='MCMC Samples Histogram')
plt.plot(x_grid, unnormalized_target(x_grid) / 3.03, 'r-', linewidth=2.5, label='Normalized Target PDF')
plt.title("Metropolis-Hastings MCMC Sampling from Unnormalized Bimodal Target")
plt.xlabel("x"); plt.ylabel("Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 4. Summary & Key Takeaways

1. **Monte Carlo Integration** approximates complex high-dimensional expectations via empirical sample averages $\frac{1}{N}\sum f(x_i)$.
2. **Importance Sampling** reweights samples from a proposal distribution $q(x)$ using weights $w = p/q$.
3. **MCMC (Metropolis-Hastings)** samples from arbitrary unnormalized probability densities by constructing a stationary Markov Chain, forming the mathematical backbone of Bayesian inference and generative modeling.
